# tfmstudy — Colab runner

Runs one job of the TimesFM 3 zero-shot study and writes results to Google Drive.

1. **Runtime → Change runtime type → T4 GPU** (or connect without GPU for the CPU job).
2. **Runtime → Run all**.

With a GPU the job is read from `MyDrive/timesfm-zeroshot-study/job.json`, without one from `job_cpu.json`
(`{"job": ..., "extra_args": ...}`), so a GPU session and a CPU session can run side by side on different models.
The code comes from the newest `code/tfmstudy-*.zip`, so this notebook never needs editing. Results land in
`MyDrive/timesfm-zeroshot-study/results/<job>/`. From another Google account: share the folder with it and add a
shortcut named `timesfm-zeroshot-study` to its My Drive.

In [ ]:
ROOT = "/content/drive/MyDrive/timesfm-zeroshot-study"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json, os, shutil, subprocess
assert os.path.isdir(ROOT), f'{ROOT} not found: add the shared folder (or a shortcut to it) to My Drive'
HAS_GPU = shutil.which('nvidia-smi') is not None and subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo 'no GPU: CPU job'
job_file = f"{ROOT}/job.json" if HAS_GPU else f"{ROOT}/job_cpu.json"
job = json.load(open(job_file)) if os.path.exists(job_file) else {}
JOB = job.get("job", "smoke")
EXTRA_ARGS = job.get("extra_args", "")
print(f"{os.path.basename(job_file)} -> job: {JOB}   extra args: {EXTRA_ARGS!r}")

In [ ]:
import glob, hashlib, os, zipfile
zips = sorted(glob.glob(f"{ROOT}/code/tfmstudy-*.zip"), key=os.path.getmtime)
assert zips, 'no code bundle found on Drive'
BUNDLE = zips[-1]
# The bundle name ends with the first 12 hex digits of its sha256: a corrupted upload stops here.
expected = os.path.basename(BUNDLE).rsplit('-', 1)[-1].removesuffix('.zip')
actual = hashlib.sha256(open(BUNDLE, 'rb').read()).hexdigest()[:12]
assert actual == expected, f'bundle hash mismatch: {actual} != {expected}'
print('using', BUNDLE, 'sha256 ok')
!rm -rf /content/src && mkdir -p /content/src
zipfile.ZipFile(BUNDLE).extractall('/content/src')

In [ ]:
!pip install -q -r /content/src/requirements-colab.txt 2>&1 | tail -3
!pip list 2>/dev/null | grep -iE '^(torch|timesfm|chronos-forecasting|tirex-2|statsforecast) '

In [ ]:
import shlex
OUT = f"{ROOT}/results/{JOB}"
!cd /content/src && python -m tfmstudy.{JOB} --out {shlex.quote(OUT)} {EXTRA_ARGS} 2>&1 | tee /content/run.log
!mkdir -p "{OUT}" && cp /content/run.log "{OUT}/run_$(date -u +%Y%m%dT%H%M%SZ).log"